# `klygo.config` (`config.py`): Stateful `Config` manager

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Config.ipynb)


Manage a configuration file through a stateful object interface supporting `.create_default()`, `.read()`, `.get()`, `.set()`, `.has()`, `.delete()`, `.merge()`, `.update()`, `.to_dict()`, `.to_json()`, and `.export_file()`.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-35xnvnkx


## Jump to an API

[`Config`](#Config)

<a id="Config"></a>

### `config.Config`

Manage one configuration file through a stateful object interface.

**Signature:** `config.Config(config_path: str | pathlib.Path)`

In [2]:
from klygo import Config, models

manager = Config.create_default(root / "settings.yaml", {"default": {"root": str(root)}}, overwrite=True, verbose=False)
print("Bound path:", files.name(manager.config_path))
manager.set("model.batch", 32)
manager.update({"model": {"epochs": 20}})
manager.merge({"seed": 123})
print("Has seed:", manager.has("seed"), "| batch:", manager.get("model.batch"))
print("Deleted lr:", manager.delete("model.lr"))
print("JSON preview:", manager.to_json(indent=2)[:80], "...")
exported = manager.export_file("exported_settings", ".toml", output_dir=root, overwrite=True, verbose=False)
print("Exported:", files.name(exported), "| exists:", files.is_file(exported))

Bound path: settings.yaml
Has seed: True | batch: 32
Deleted lr: True
JSON preview: {
  "default": {
    "root": "C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-confi ...
Exported: exported_settings.toml | exists: True


In [3]:
# Persist exact model metadata without injecting general Klygo defaults
flags = models.flags(model=False, processor=True, post=True)
schema = models.metadata(flags)
meta_manager = Config.create_default(root / "model.toml", metadata=schema, overwrite=True, verbose=False)
print("Exact metadata keys:", sorted(meta_manager.to_dict().keys()))
print("flags.model:", meta_manager.get("flags.model"), "| class:", meta_manager.get("class"))

Exact metadata keys: ['class', 'details', 'flags', 'priority']
flags.model: False | class: None


In [4]:
# Error examples: passing both default_data and metadata, or negative JSON indent
try:
    Config.create_default(root / "bad.yaml", default_data={}, metadata=schema, verbose=False)
except ValueError as exc:
    print("ValueError ->", exc)

try:
    manager.to_json(indent=-1)
except ValueError as exc:
    print("ValueError ->", exc)

ValueError -> default_data and metadata are mutually exclusive
ValueError -> indent must be a non-negative integer


## Put it together

In [5]:
from klygo import Config

managed_path = root / "managed_experiment.yaml"
mgr = Config.create_default(managed_path, {"model": {"name": "vit", "lr": 0.001}}, overwrite=True, verbose=False)
mgr.set("model.lr", 0.0005)
mgr.update({"training": {"epochs": 30}})
snap = mgr.export_file("snapshot", ".json", output_dir=root, overwrite=True, verbose=False)
print("Snapshot:", files.name(snap), "| epochs:", config.load(snap, verbose=False).training.epochs)

Snapshot: snapshot.json | epochs: 30


## Remember

- `Config` binds a file path and updates its in-memory state when you call `.read()`, `.set()`, `.delete()`, `.merge()`, or `.update()`, persisting changes when you call `.export_file()` or `Config.create_default()`.
- Next in `klygo.config`: start over at [Document_Config_Io.ipynb](Document_Config_Io.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).